In [ ]:
!pip install segmentation-models-pytorch
!pip install segment-anything ultralytics
!pip install albumentations

!pip install --upgrade sympy
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install segment-anything ultralytics
!pip install pandas seaborn scikit-learn albumentations matplotlib

In [ ]:
!pip install git+https://github.com/facebookresearch/segment-anything-2.git

from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

In [ ]:
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
!mkdir -p checkpoints
!wget -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt


In [25]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [26]:
import os, sys, json, time, logging, warnings
from pathlib import Path
import shutil
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, asdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
from segment_anything import sam_model_registry, SamPredictor, SamAutomaticMaskGenerator
from ultralytics import YOLO

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

In [27]:
@dataclass
class Config:
    dataset_path: str = "./kvasir-capsule-seg"
    output_dir: str = "./output"
    sam_checkpoint_path: str = "sam_vit_b_01ec64.pth"
    sam2_checkpoint_path: str = "./checkpoints/sam2_hiera_large.pt"
    sam2_config: str = "sam2_hiera_l.yaml"
    img_size: Tuple[int, int] = (256, 256)
    batch_size: int = 8
    num_epochs: int = 100
    learning_rate: float = 1e-4
    weight_decay: float = 1e-5
    early_stopping_patience: int = 15
    test_size: float = 0.2
    val_size: float = 0.15
    random_seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers: int = 4
    pin_memory: bool = True
    aug_probability: float = 0.5
    dice_threshold: float = 0.75

    def __post_init__(self):
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.run_dir = Path(self.output_dir) / f"run_{self.timestamp}"
        self.run_dir.mkdir(parents=True, exist_ok=True)
        (self.run_dir / "models").mkdir(exist_ok=True)
        (self.run_dir / "visualizations").mkdir(exist_ok=True)
        (self.run_dir / "metrics").mkdir(exist_ok=True)
        (self.run_dir / "logs").mkdir(exist_ok=True)

In [28]:
def setup_logger(config: Config) -> logging.Logger:
    logger = logging.getLogger("PolypSegmentation")
    logger.setLevel(logging.DEBUG)
    fh = logging.FileHandler(config.run_dir / "logs" / "pipeline.log")
    fh.setLevel(logging.DEBUG)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)
    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger

The `AugmentationPipeline` class contains two methods: `get_train_transforms` and `get_val_transforms`.
- `get_train_transforms` defines a sequence of augmentation operations applied to the training data, including flips, rotations, shifts, scale, elastic transformations, brightness/contrast adjustments, hue/saturation/value shifts, and noise/blur. It also resizes images to the specified `img_size` in the configuration and normalizes them.
- `get_val_transforms` defines the transformations for the validation data, which typically only includes resizing and normalization to ensure consistent input size and scale without introducing random variations.

The `albumentations` library is used to define the image augmentation pipelines.
- `A.Compose` is used to chain multiple transformations together.
- `ToTensorV2()` converts the augmented NumPy arrays or PIL images into PyTorch tensors and scales pixel values to the range [0, 1].
- `A.Normalize` normalizes the pixel values using the specified mean and standard deviation, which are standard values for images trained on ImageNet.

Data augmentation is a crucial technique in training deep learning models, especially when dealing with limited datasets. By artificially increasing the diversity of the training data, it helps prevent overfitting and improves the model's ability to generalize to unseen images.

This cell does not produce any direct visual output.
It defines the `AugmentationPipeline` class, which will be used by the `DataModule` to apply transformations to the image and mask data during dataset loading.

In [29]:
class AugmentationPipeline:
    def __init__(self, config: Config):
        self.config = config
        self.logger = logging.getLogger("PolypSegmentation.Augmentation")

    def get_train_transforms(self) -> A.Compose:
        return A.Compose([
            A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.Rotate(limit=30, p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=0.5),
            A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3), A.GaussianBlur(blur_limit=(3, 5), p=0.2),
            A.Resize(*self.config.img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    def get_val_transforms(self) -> A.Compose:
        return A.Compose([
            A.Resize(*self.config.img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

This cell defines two classes: `PolypDataset` and `DataModule`. These classes are essential for handling the loading, preprocessing, and splitting of the polyp segmentation dataset.

- The `PolypDataset` class is a custom PyTorch `Dataset` that loads image and mask pairs, applies transformations (defined by the `AugmentationPipeline`), and returns them as PyTorch tensors. It handles reading images and masks using OpenCV and converting the mask to a binary format.
- The `DataModule` class orchestrates the data preparation process. It takes the dataset path from the configuration, finds all image and mask files, and splits them into training, validation, and test sets using `train_test_split` from scikit-learn. It then creates PyTorch `DataLoader` instances for each split, which are used to efficiently load data in batches during training and evaluation.


In [30]:
class PolypDataset(Dataset):
    def __init__(self, image_paths: List[Path], mask_paths: List[Path], transforms: Optional[A.Compose] = None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transforms = transforms
        assert len(image_paths) == len(mask_paths)

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        image = cv2.cvtColor(cv2.imread(str(self.image_paths[idx])), cv2.COLOR_BGR2RGB)
        mask = (cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image, mask = augmented['image'], augmented['mask']

        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask).unsqueeze(0).float()
        elif isinstance(mask, torch.Tensor) and len(mask.shape) == 2:
            mask = mask.unsqueeze(0).float()

        return image, mask


class DataModule:
    def __init__(self, config: Config):
        self.config = config
        self.logger = logging.getLogger("PolypSegmentation.DataModule")
        self.aug_pipeline = AugmentationPipeline(config)

    def prepare_data(self) -> Dict[str, DataLoader]:
        self.logger.info("="*60 + "\nSTAGE 1: DATA PREPARATION\n" + "="*60)
        data_path = Path(self.config.dataset_path)
        image_paths = sorted(list((data_path / "images").glob("*.jpg")))
        mask_paths = sorted(list((data_path / "masks").glob("*.jpg")))

        train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(
            image_paths, mask_paths, test_size=self.config.test_size + self.config.val_size, random_state=self.config.random_seed)
        val_imgs, test_imgs, val_masks, test_masks = train_test_split(
            temp_imgs, temp_masks, test_size=self.config.val_size/(self.config.test_size + self.config.val_size), random_state=self.config.random_seed)

        self.logger.info(f"Train={len(train_imgs)}, Val={len(val_imgs)}, Test={len(test_imgs)}")

        # Store splits for YOLO conversion
        self.train_imgs, self.train_masks = train_imgs, train_masks
        self.val_imgs, self.val_masks = val_imgs, val_masks
        self.test_imgs, self.test_masks = test_imgs, test_masks

        return {
            'train': DataLoader(PolypDataset(train_imgs, train_masks, self.aug_pipeline.get_train_transforms()),
                               batch_size=self.config.batch_size, shuffle=True, num_workers=self.config.num_workers, pin_memory=self.config.pin_memory),
            'val': DataLoader(PolypDataset(val_imgs, val_masks, self.aug_pipeline.get_val_transforms()),
                             batch_size=self.config.batch_size, shuffle=False, num_workers=self.config.num_workers, pin_memory=self.config.pin_memory),
            'test': DataLoader(PolypDataset(test_imgs, test_masks, self.aug_pipeline.get_val_transforms()),
                              batch_size=1, shuffle=False, num_workers=self.config.num_workers, pin_memory=self.config.pin_memory)
        }

In [31]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()
        self.pool = nn.MaxPool2d(2, 2)

        for feature in features:
            self.encoder.append(self._block(in_channels, feature))
            in_channels = feature

        self.bottleneck = self._block(features[-1], features[-1] * 2)

        for feature in reversed(features):
            self.decoder.append(nn.ConvTranspose2d(feature*2, feature, 2, 2))
            self.decoder.append(self._block(feature*2, feature))

        self.final_conv = nn.Conv2d(features[0], out_channels, 1)

    def _block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, 1, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )

    def forward(self, x):
        skip_connections = []
        for encode in self.encoder:
            x = encode(x)
            skip_connections.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]
        for idx in range(0, len(self.decoder), 2):
            x = self.decoder[idx](x)
            skip = skip_connections[idx // 2]
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.decoder[idx + 1](x)
        return self.final_conv(x)



This cell defines the `UNet` class, a classic convolutional neural network architecture widely used for image segmentation tasks. It serves as one of the models to be trained and evaluated in this notebook for polyp segmentation.

**Detailed explanation:**
- The `UNet` architecture consists of a contracting path (encoder) and an expansive path (decoder), which gives it a U-shape.
- The encoder progressively downsamples the input image, capturing context, while the decoder upsamples the feature maps, allowing for precise localization of the segmented object.
- Crucially, `skip connections` are used to transfer feature maps from the encoder to the corresponding layers in the decoder. These connections help the decoder recover spatial information lost during downsampling, leading to more accurate segmentation boundaries.
- The `_block` method defines a standard convolutional block used throughout the network, consisting of two convolutional layers, batch normalization, and ReLU activation.

Partitions:
- The model is implemented using PyTorch's `torch.nn` module.
- `nn.Conv2d` is used for convolutional layers.
- `nn.BatchNorm2d` is used for batch normalization, which helps stabilize training.
- `nn.ReLU` is the activation function.
- `nn.MaxPool2d` is used for downsampling in the encoder.
- `nn.ConvTranspose2d` is used for upsampling in the decoder.
- `torch.cat` is used to concatenate the decoder output with the skip connections.
- `F.interpolate` is used for resizing feature maps if their shapes don't match for concatenation.

UNet is a foundational architecture for medical image segmentation and provides a strong baseline for comparison with other models like SAM and YOLO. Its skip connections are particularly effective for tasks requiring precise pixel-wise prediction.


The MetricsCalculator class is designed to calculate these metrics for comparing predicted segmentation masks against ground truth masks.
- _bin is a static helper method used internally to convert the model's output logits (raw predictions) into binary masks by applying a sigmoid activation and thresholding at 0.5.

- dice_coefficient computes the Sørensen–Dice coefficient, a measure of similarity between two binary masks. It's commonly used in medical image segmentation.

- iou_score calculates the Intersection over Union (IoU), also known as the Jaccard index, another common metric for evaluating the overlap between predicted and ground truth masks.

- pixel_accuracy calculates the proportion of correctly classified pixels (both true positives and true negatives) relative to the total number of pixels.

A small smooth value is added in Dice and IoU calculations to prevent division by zero, especially when dealing with empty masks.


In [32]:
class MetricsCalculator:
    @staticmethod
    def _bin(pred_logits): return (torch.sigmoid(pred_logits) > 0.5).float()

    @staticmethod
    def dice_coefficient(pred_logits, target, smooth=1e-7):
        pred = MetricsCalculator._bin(pred_logits)
        intersection = (pred * target).sum()
        return ((2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)).item()

    @staticmethod
    def iou_score(pred_logits, target, smooth=1e-7):
        pred = MetricsCalculator._bin(pred_logits)
        intersection = (pred * target).sum()
        union = pred.sum() + target.sum() - intersection
        return ((intersection + smooth) / (union + smooth)).item()

    @staticmethod
    def pixel_accuracy(pred_logits, target):
        pred = MetricsCalculator._bin(pred_logits)
        return ((pred == target).float().sum() / torch.numel(pred)).item()

This cell defines the Trainer class, which encapsulates the core logic for training a PyTorch segmentation model. It handles the training loop, validation, model saving, and early stopping.


In [33]:
class Trainer:
    def __init__(self, model, model_name, config, criterion, optimizer, scheduler=None):
        self.model = model.to(config.device)
        self.model_name = model_name
        self.config = config
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.logger = logging.getLogger(f"PolypSegmentation.{model_name}")
        self.metrics_calc = MetricsCalculator()
        self.best_val_dice = 0.0
        self.patience_counter = 0
        self.training_history = []

    def train_epoch(self, dataloader):
        self.model.train()
        total_loss = 0.0
        for images, masks in tqdm(dataloader, desc=f"Training {self.model_name}"):
            images, masks = images.to(self.config.device), masks.to(self.config.device)
            outputs = self.model(images)
            loss = self.criterion(outputs, masks)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        return total_loss / len(dataloader)

    def validate(self, dataloader):
        self.model.eval()
        total_loss = 0.0
        dice_scores, iou_scores, pixel_accs = [], [], []
        with torch.no_grad():
            for images, masks in tqdm(dataloader, desc=f"Validating {self.model_name}"):
                images, masks = images.to(self.config.device), masks.to(self.config.device)
                outputs = self.model(images)
                total_loss += self.criterion(outputs, masks).item()
                for i in range(outputs.size(0)):
                    dice_scores.append(self.metrics_calc.dice_coefficient(outputs[i], masks[i]))
                    iou_scores.append(self.metrics_calc.iou_score(outputs[i], masks[i]))
                    pixel_accs.append(self.metrics_calc.pixel_accuracy(outputs[i], masks[i]))
        return total_loss / len(dataloader), {'dice': np.mean(dice_scores), 'iou': np.mean(iou_scores), 'pixel_acc': np.mean(pixel_accs)}

    def fit(self, train_loader, val_loader):
        self.logger.info("="*60 + f"\nTRAINING: {self.model_name}\n" + "="*60)
        start_time = time.time()
        for epoch in range(self.config.num_epochs):
            train_loss = self.train_epoch(train_loader)
            val_loss, val_metrics = self.validate(val_loader)
            if self.scheduler:
                self.scheduler.step(val_loss) if isinstance(self.scheduler, ReduceLROnPlateau) else self.scheduler.step()

            self.logger.info(f"Epoch {epoch+1}/{self.config.num_epochs} | Train Loss: {train_loss:.4f} | Val Dice: {val_metrics['dice']:.4f}")

            if val_metrics['dice'] > self.best_val_dice:
                self.best_val_dice = val_metrics['dice']
                self.patience_counter = 0
                self.save_checkpoint('best')
            else:
                self.patience_counter += 1

            if self.patience_counter >= self.config.early_stopping_patience:
                self.logger.info(f"Early stopping at epoch {epoch+1}")
                break

        return {'best_dice': self.best_val_dice, 'training_time': time.time() - start_time, 'epochs_trained': epoch+1}

    def save_checkpoint(self, tag):
        torch.save({'model_state_dict': self.model.state_dict(), 'best_val_dice': self.best_val_dice},
                  self.config.run_dir / "models" / f"{self.model_name}_{tag}.pth")

    def load_checkpoint(self, tag):
        checkpoint = torch.load(self.config.run_dir / "models" / f"{self.model_name}_{tag}.pth",
                               map_location=self.config.device, weights_only=False)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.best_val_dice = checkpoint['best_val_dice']

- YOLOSegmentationWrapper is a wrapper around the ultralytics.YOLO class, providing methods for initializing, training, predicting, and validating YOLO segmentation models using their native API. It includes logging for better debugging.

- YOLOTrainer is a specialized trainer class for YOLO models, utilizing the YOLOSegmentationWrapper. It manages the training process for YOLO models, including calling the native training method and handling checkpoint loading.

- YOLODatasetConverter is responsible for converting the dataset from the current format (images and binary masks) to the YOLO segmentation format, which involves creating a directory structure and generating text files with normalized polygon coordinates for each mask.

- YOLOEvaluator is designed to evaluate the performance of a trained YOLO segmentation model on a test set, calculating metrics like Dice, IoU, and Pixel Accuracy, and visualizing predictions.


In [34]:
class YOLOSegmentationWrapper:
    """Wrapper for native YOLO segmentation training with enhanced debugging."""
    def __init__(self, model_name, config):
        self.config = config
        self.model_name = model_name
        self.logger = logging.getLogger(f"PolypSegmentation.{model_name}")
        self.device = config.device

        self.logger.info(f"Initializing YOLO model with name: '{model_name}'")

        # Attempt to load the model either as a pretrained model name or as a file path
        try:
            # First check if model_name is a path to an existing file
            if Path(model_name).is_file():
                self.logger.info(f"Loading model from file: {model_name}")
                self.yolo = YOLO(model_name)
            else:
                # Otherwise, try to load as a pretrained model from the Ultralytics Hub
                self.logger.info(f"Attempting to load pretrained model: '{model_name}' from Ultralytics Hub.")
                self.yolo = YOLO(f"{model_name}.pt")

                # Verify that the model actually loaded segmentation weights
                if hasattr(self.yolo.model, 'nc') and self.yolo.model.nc == 1:
                    self.logger.info("Model loaded successfully and appears to be configured for single-class segmentation.")
                else:
                    self.logger.warning(f"Model loaded, but number of classes (nc={getattr(self.yolo.model, 'nc', 'N/A')}) is unexpected for polyp segmentation (expected 1).")

        except Exception as e:
            self.logger.error(f" FAILED to initialize YOLO model '{model_name}'. Error: {e}", exc_info=True)
            raise

    def train_native(self, train_data_yaml, epochs, img_size):
        """Train using YOLO's native training API"""
        self.logger.info(f"Starting native training for {self.model_name}...")
        results = self.yolo.train(
            data=train_data_yaml,
            epochs=epochs,
            imgsz=img_size,
            batch=4,
            device=self.device,
            project=str(self.config.run_dir / "models"),
            name=self.model_name,
            patience=self.config.early_stopping_patience,
            save=True,
            plots=True,
            val=True
        )
        self.logger.info(f"Training finished. Results saved to: {results.save_dir}")
        return results

    def predict(self, image, conf=0.01, iou=0.7):
        try:
            results = self.yolo.predict(
                image,
                device=self.device,
                conf=conf,
                iou=iou,
                retina_masks=True,  # flag for pixel-level masks
                verbose=False
            )
            pred_result = results[0]

            if pred_result.masks is not None:
                # pred_result.masks.data: Tensor [N, H, W] (see official docs)
                mask_tensor = pred_result.masks.data.float()
                combined_mask = mask_tensor.amax(dim=0, keepdim=True)  # merge all polygons
                return combined_mask.cpu()
            else:
                self.logger.debug("No masks returned after thresholding.")
                return None
        except Exception as e:
            self.logger.error(f"Error during prediction: {e}", exc_info=True)
            return None

    def val(self, data_yaml):
        """Validate using YOLO's native API"""
        self.logger.info(f"Starting validation for {self.model_name}...")
        metrics = self.yolo.val(data=data_yaml, device=self.device)
        self.logger.info(f"Validation finished. mAP50-seg: {getattr(metrics, 'map50', 'N/A')}")
        return metrics



class YOLOTrainer:
    """Trainer for YOLO models using native API with enhanced debugging."""
    def __init__(self, model_wrapper, model_name, config, data_yaml):
        self.model = model_wrapper
        self.model_name = model_name
        self.config = config
        self.data_yaml = data_yaml
        self.logger = logging.getLogger(f"PolypSegmentation.{model_name}")
        self.metrics_calc = MetricsCalculator()
        self.best_val_dice = 0.0
        self.training_history = []
        self.results_save_dir = None

    def fit(self):
        """Train using YOLO's native training"""
        self.logger.info("="*60 + f"\nTRAINING: {self.model_name}\n" + "="*60)
        start_time = time.time()

        # Train with YOLO native API
        results = self.model.train_native(
            train_data_yaml=self.data_yaml,
            epochs=self.config.num_epochs,
            img_size=self.config.img_size[0]
        )

        # results.save_dir contains the exact path to the folder where YOLO saved the model.
        self.results_save_dir = Path(results.save_dir)
        self.logger.info(f"Training results are stored in: {self.results_save_dir}")

        training_time = time.time() - start_time

        # Get validation metrics from YOLO
        val_metrics = self.model.val(self.data_yaml)
        self.best_val_dice = val_metrics.seg.map if hasattr(val_metrics, 'seg') else 0.0

        self.logger.info(f"Training completed in {training_time:.2f}s. Best mAP-seg: {self.best_val_dice:.4f}")

        return {
            'best_dice': self.best_val_dice,
            'training_time': training_time,
            'epochs_trained': self.config.num_epochs
        }

    def save_checkpoint(self, tag):
        """YOLO saves checkpoints automatically"""
        pass

    def load_checkpoint(self, tag):
        """Load best YOLO checkpoint using the correct path from training results."""
        if self.results_save_dir is None:
            raise ValueError("Model must be trained before loading a checkpoint.")

        # --- FIX: Use the path returned by YOLO rather than constructing it ourselves ---
        best_model_path = self.results_save_dir / "weights" / "best.pt"
        self.logger.info(f"Attempting to load best checkpoint from: {best_model_path}")

        if best_model_path.exists():
            try:
                self.model.yolo = YOLO(str(best_model_path))
                self.logger.info(f"Successfully loaded best checkpoint from {best_model_path}")
            except Exception as e:
                self.logger.error(f"Failed to load best checkpoint. Error: {e}", exc_info=True)
                raise
        else:
            self.logger.error(f"Best checkpoint NOT FOUND at {best_model_path}.")
            raise FileNotFoundError(f"Best model checkpoint not found at {best_model_path}")

class YOLODatasetConverter:
    """Convert dataset to YOLO format"""
    def __init__(self, config):
        self.config = config
        self.logger = logging.getLogger("PolypSegmentation.YOLOConverter")

    def convert_to_yolo_format(self, train_imgs, train_masks, val_imgs, val_masks, test_imgs, test_masks):
        """Convert Kvasir dataset to YOLO segmentation format"""
        yolo_dir = self.config.run_dir / "yolo_dataset"
        (yolo_dir / "images" / "train").mkdir(parents=True, exist_ok=True)
        (yolo_dir / "images" / "val").mkdir(parents=True, exist_ok=True)
        (yolo_dir / "images" / "test").mkdir(parents=True, exist_ok=True)
        (yolo_dir / "labels" / "train").mkdir(parents=True, exist_ok=True)
        (yolo_dir / "labels" / "val").mkdir(parents=True, exist_ok=True)
        (yolo_dir / "labels" / "test").mkdir(parents=True, exist_ok=True)

        self.logger.info("Converting dataset to YOLO format...")

        # Convert train
        self._convert_split(train_imgs, train_masks, yolo_dir / "images" / "train", yolo_dir / "labels" / "train")
        # Convert val
        self._convert_split(val_imgs, val_masks, yolo_dir / "images" / "val", yolo_dir / "labels" / "val")
        # Convert test
        self._convert_split(test_imgs, test_masks, yolo_dir / "images" / "test", yolo_dir / "labels" / "test")

        # Create data.yaml
        data_yaml = yolo_dir / "data.yaml"
        with open(data_yaml, 'w') as f:
            f.write(f"""
                    path: {yolo_dir.absolute()}
                    train: images/train
                    val: images/val
                    test: images/test

                    nc: 1
                    names: ['polyp']
                    """)

        self.logger.info(f"YOLO dataset created at {yolo_dir}")
        return str(data_yaml)

    def _convert_split(self, image_paths, mask_paths, img_dir, label_dir):
        for img_path, mask_path in zip(image_paths, mask_paths):
            shutil.copy(img_path, img_dir / img_path.name)
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                print(f"[WARN] Cannot read mask: {mask_path}")
                continue
            h, w = mask.shape[:2]
            mask_bin = (mask > 127).astype(np.uint8)
            contours, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            label_file = label_dir / f"{img_path.stem}.txt"
            with open(label_file, "w") as f:
                for contour in contours:
                    if cv2.contourArea(contour) < 10:
                        continue
                    contour = contour.squeeze(1)
                    normalized = contour.astype(np.float32)
                    normalized[:, 0] /= w
                    normalized[:, 1] /= h
                    normalized = np.clip(normalized, 0, 1)
                    points = " ".join(f"{x:.6f} {y:.6f}" for x, y in normalized)
                    f.write(f"0 {points}\n")
            if not contours:
                label_file.touch()


class YOLOEvaluator:
    """Evaluator for YOLO models"""
    def __init__(self, config):
        self.config = config
        self.logger = logging.getLogger("PolypSegmentation.YOLOEvaluator")
        self.metrics_calc = MetricsCalculator()

    def evaluate(self, model_wrapper, model_name, test_loader):
        """Evaluate YOLO model on test set"""
        self.logger.info(f"Evaluating {model_name} on test set")

        dice_scores, iou_scores, pixel_accs = [], [], []

        for images, masks in tqdm(test_loader, desc=f"Testing {model_name}"):
            # Get image as numpy
            image_np = images[0].cpu().permute(1, 2, 0).numpy()
            mask_np = masks[0, 0].cpu().numpy()

            # Denormalize
            mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
            image_np = np.clip(std * image_np + mean, 0, 1)
            image_np = (image_np * 255).astype(np.uint8)

            # Predict with YOLO
            pred_masks = model_wrapper.predict(image_np)

            if pred_masks is not None and len(pred_masks) > 0:
                # Take first mask and resize to match ground truth
                pred_mask = pred_masks[0].cpu().numpy()
                pred_mask = cv2.resize(pred_mask, (mask_np.shape[1], mask_np.shape[0]))
                pred_mask = torch.from_numpy(pred_mask).unsqueeze(0).float()
                mask_tensor = torch.from_numpy(mask_np).unsqueeze(0).float()

                # Calculate metrics (pred_mask is already probabilities from YOLO)
                pred_logits = torch.logit(torch.clamp(pred_mask, 1e-7, 1-1e-7))
                dice_scores.append(self.metrics_calc.dice_coefficient(pred_logits, mask_tensor))
                iou_scores.append(self.metrics_calc.iou_score(pred_logits, mask_tensor))
                pixel_accs.append(self.metrics_calc.pixel_accuracy(pred_logits, mask_tensor))
            else:
                # No detection - all zeros
                dice_scores.append(0.0)
                iou_scores.append(0.0)
                pixel_accs.append(np.mean(mask_np == 0))

        return {
            'mean_dice': np.mean(dice_scores),
            'std_dice': np.std(dice_scores),
            'mean_iou': np.mean(iou_scores),
            'std_iou': np.std(iou_scores),
            'mean_pixel_acc': np.mean(pixel_accs),
            'std_pixel_acc': np.std(pixel_accs)
        }

    def visualize_predictions(self, model_wrapper, model_name, test_loader, num_samples=10):
        """Visualize YOLO predictions"""
        self.logger.info(f"Generating predictions visualization for {model_name}")

        fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))

        sample_count = 0
        for images, masks in test_loader:
            if sample_count >= num_samples:
                break

            image_np = images[0].cpu().permute(1,2,0).numpy()
            mask_np = masks[0,0].cpu().numpy()

            mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
            image_vis = np.clip(std * image_np + mean, 0, 1)
            image_np_uint8 = (image_vis * 255).astype(np.uint8)

            pred_masks = model_wrapper.predict(image_np_uint8)

            if pred_masks is not None and len(pred_masks) > 0:
                pred = cv2.resize(pred_masks[0].cpu().numpy(), (mask_np.shape[1], mask_np.shape[0]))
            else:
                pred = np.zeros_like(mask_np)

            axes[sample_count, 0].imshow(image_vis)
            axes[sample_count, 0].axis('off')
            axes[sample_count, 1].imshow(mask_np, cmap='gray')
            axes[sample_count, 1].axis('off')
            axes[sample_count, 2].imshow(pred, cmap='hot', vmin=0, vmax=1)
            axes[sample_count, 2].axis('off')
            axes[sample_count, 3].imshow((pred > 0.5).astype(np.float32), cmap='gray')
            axes[sample_count, 3].axis('off')

            sample_count += 1

        plt.tight_layout()
        plt.savefig(self.config.run_dir / "visualizations" / f"{model_name}_predictions.png", dpi=150)
        plt.show()
        plt.close()

- SAMSegmentationWrapper wraps the fine-tuned SAM model, adding an adapter layer to produce a single-channel output suitable for binary segmentation. It allows for freezing the image encoder during fine-tuning.

- SAMTrainer is a specialized trainer for the fine-tuned SAM model, incorporating features like gradient accumulation and mixed precision training (AMP) for efficient training.

- SAMOneShotWrapper uses the original SAM model's automatic mask generation capability to perform one-shot segmentation without further training. It selects the best mask based on predicted IoU.

- SAMOneShotEvaluator evaluates the performance of the SAM One-Shot approach on the test set and visualizes predictions.

- SAM2OneShotWrapper is similar to SAMOneShotWrapper but adapted for the SAM-2 model, utilizing SAM2AutomaticMaskGenerator for one-shot segmentation.

- SAM2OneShotEvaluator evaluates the performance of the SAM-2 One-Shot approach on the test set and visualizes its predictions.


In [35]:
class SAMSegmentationWrapper(nn.Module):
    def __init__(self, sam_checkpoint, model_type, freeze_encoder, config, target_size=1024):
        super().__init__()
        self.config = config
        self.freeze_encoder = freeze_encoder
        self.target_size = target_size

        self.sam = sam_model_registry[model_type](checkpoint=sam_checkpoint).to(config.device)

        if freeze_encoder:
            for param in self.sam.image_encoder.parameters():
                param.requires_grad = False
            self.sam.image_encoder.eval()  # keep BN in eval mode

        self.adapter = nn.Sequential(
            nn.Conv2d(256, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=1),
        )

    def forward(self, x):
        B, _, H, W = x.shape
        x_resized = F.interpolate(x, size=(self.target_size, self.target_size),
                                  mode="bilinear", align_corners=False)

        if self.freeze_encoder:
            with torch.no_grad():
                features = self.sam.image_encoder(x_resized)
            features = features.detach()             # optional but explicit
        else:
            features = self.sam.image_encoder(x_resized)

        output = self.adapter(features)
        return F.interpolate(output, size=(H, W), mode="bilinear", align_corners=False)

class SAMTrainer(Trainer):
    def __init__(self, model, model_name, config, criterion, optimizer,
                 scheduler=None, gradient_accumulation_steps=4, use_amp=True):
        super().__init__(model, model_name, config, criterion, optimizer, scheduler)
        self.gradient_accumulation_steps = gradient_accumulation_steps
        self.use_amp = use_amp and config.device.startswith("cuda")
        self.scaler = torch.cuda.amp.GradScaler(enabled=self.use_amp)

    def train_epoch(self, dataloader):
        self.model.train()
        if getattr(self.model, "freeze_encoder", False):
            self.model.sam.image_encoder.eval()

        total_loss = 0.0
        self.optimizer.zero_grad(set_to_none=True)

        for batch_idx, (images, masks) in enumerate(
                tqdm(dataloader, desc=f"Training {self.model_name}"), start=1):
            images = images.to(self.config.device, non_blocking=True)
            masks = masks.to(self.config.device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=self.use_amp):
                logits = self.model(images)
                loss = self.criterion(logits, masks)

            loss_value = loss.item()
            total_loss += loss_value

            loss = loss / self.gradient_accumulation_steps

            if self.use_amp:
                self.scaler.scale(loss).backward()
            else:
                loss.backward()

            if batch_idx % self.gradient_accumulation_steps == 0 or batch_idx == len(dataloader):
                if self.use_amp:
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    self.optimizer.step()

                self.optimizer.zero_grad(set_to_none=True)

        return total_loss / len(dataloader)

class SAMOneShotWrapper:
    def __init__(self, config: Config, model_type: str = "vit_b", generator_kwargs: Optional[Dict[str, Any]] = None):
        self.config = config
        checkpoint = config.sam_checkpoint_path
        if not Path(checkpoint).exists():
            raise FileNotFoundError(f"SAM checkpoint not found at {checkpoint}")

        self.logger = logging.getLogger("PolypSegmentation.SAMOneShot")
        self.logger.info("Initializing SAM One-Shot wrapper")

        self.sam = sam_model_registry[model_type](checkpoint=checkpoint).to(config.device)
        self.sam.eval()

        default_generator_kwargs = dict(
            points_per_side=16,
            points_per_batch=64,
            pred_iou_thresh=0.9,
            stability_score_thresh=0.92,
            crop_n_layers=0,
        )
        if generator_kwargs:
            default_generator_kwargs.update(generator_kwargs)

        self.mask_generator = SamAutomaticMaskGenerator(model=self.sam, **default_generator_kwargs)

    @torch.no_grad()
    def predict(self, image_np: np.ndarray) -> Optional[torch.Tensor]:
        masks = self.mask_generator.generate(image_np)
        if not masks:
            return None

        best_mask = max(masks, key=lambda x: x.get("predicted_iou", 0))
        mask = best_mask["segmentation"].astype(np.float32)
        return torch.from_numpy(mask).unsqueeze(0)

class SAMOneShotEvaluator:
    def __init__(self, config: Config):
        self.config = config
        self.logger = logging.getLogger("PolypSegmentation.SAMOneShotEvaluator")
        self.metrics_calc = MetricsCalculator()
        self.wrapper = SAMOneShotWrapper(config)

    def evaluate(self, test_loader: DataLoader) -> Dict[str, float]:
        self.logger.info("Evaluating SAM One-Shot on test set")
        dice_scores, iou_scores, pixel_accs = [], [], []

        for images, masks in tqdm(test_loader, desc="Testing SAM One-Shot"):
            image_np = images[0].cpu().permute(1, 2, 0).numpy()
            mask_np = masks[0, 0].cpu().numpy()

            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            image_rgb = np.clip(std * image_np + mean, 0, 1)
            image_uint8 = (image_rgb * 255).astype(np.uint8)

            pred_mask = self.wrapper.predict(image_uint8)

            if pred_mask is not None:
                pred_probs = pred_mask.clamp(0, 1)
                pred_logits = torch.logit(pred_probs, eps=1e-6)
            else:
                pred_logits = torch.logit(torch.full_like(masks[0], 1e-6), eps=1e-6)

            gt_mask = masks[0]

            dice_scores.append(self.metrics_calc.dice_coefficient(pred_logits, gt_mask))
            iou_scores.append(self.metrics_calc.iou_score(pred_logits, gt_mask))
            pixel_accs.append(self.metrics_calc.pixel_accuracy(pred_logits, gt_mask))

        return {
            'mean_dice': float(np.mean(dice_scores)),
            'std_dice': float(np.std(dice_scores)),
            'mean_iou': float(np.mean(iou_scores)),
            'std_iou': float(np.std(iou_scores)),
            'mean_pixel_acc': float(np.mean(pixel_accs)),
            'std_pixel_acc': float(np.std(pixel_accs)),
        }

    def visualize_predictions(self, test_loader: DataLoader, num_samples: int = 10):
        self.logger.info("Visualizing SAM One-Shot predictions")

        fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
        if num_samples == 1:
            axes = axes.reshape(1, -1)
        col_titles = ["Input Image", "Ground Truth", "SAM Prob.", "SAM Mask"]
        for j, title in enumerate(col_titles):
            axes[0, j].set_title(title, fontsize=14, fontweight="bold")

        sample_count = 0
        for images, masks in test_loader:
            if sample_count >= num_samples:
                break

            image_np = images[0].cpu().permute(1, 2, 0).numpy()
            mask_np = masks[0, 0].cpu().numpy()

            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            image_rgb = np.clip(std * image_np + mean, 0, 1)
            image_uint8 = (image_rgb * 255).astype(np.uint8)

            pred_mask = self.wrapper.predict(image_uint8)
            if pred_mask is not None:
                pred = pred_mask.squeeze(0).numpy()
            else:
                pred = np.zeros_like(mask_np)

            axes[sample_count, 0].imshow(image_rgb)
            axes[sample_count, 1].imshow(mask_np, cmap='gray')
            axes[sample_count, 2].imshow(pred, cmap='hot', vmin=0, vmax=1)
            axes[sample_count, 3].imshow((pred > 0.5).astype(np.float32), cmap='gray')

            for j in range(4):
                axes[sample_count, j].axis('off')

            sample_count += 1

        plt.tight_layout()
        save_path = self.config.run_dir / "visualizations" / "SAM_OneShot_predictions.png"
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        self.logger.info(f"Saved SAM One-Shot visualization to {save_path}")

class SAM2OneShotWrapper:
    """SAM2 One-Shot Wrapper for automatic mask generation"""

    def __init__(self, config: Config, generator_kwargs: Optional[Dict[str, Any]] = None):
        self.config = config
        self.logger = logging.getLogger("PolypSegmentation.SAM2OneShot")
        self.logger.info("Initializing SAM2 One-Shot wrapper")

        checkpoint = config.sam2_checkpoint_path
        model_cfg = config.sam2_config

        if not Path(checkpoint).exists():
            raise FileNotFoundError(f"SAM2 checkpoint not found at {checkpoint}")

        self.sam2 = build_sam2(model_cfg, checkpoint, device=config.device)
        self.sam2.eval()

        default_generator_kwargs = dict(
            points_per_side=32,
            points_per_batch=64,
            pred_iou_thresh=0.88,
            stability_score_thresh=0.95,
            crop_n_layers=1,
            crop_n_points_downscale_factor=2,
            min_mask_region_area=100,
        )

        if generator_kwargs:
            default_generator_kwargs.update(generator_kwargs)

        self.mask_generator = SAM2AutomaticMaskGenerator(
            model=self.sam2,
            **default_generator_kwargs
        )

    @torch.no_grad()
    def predict(self, image_np: np.ndarray) -> Optional[torch.Tensor]:
        try:
            masks = self.mask_generator.generate(image_np)

            if not masks:
                self.logger.debug("No masks generated")
                return None

            best_mask = max(masks, key=lambda x: x.get("predicted_iou", 0))

            mask = best_mask["segmentation"].astype(np.float32)
            return torch.from_numpy(mask).unsqueeze(0)

        except Exception as e:
            self.logger.error(f"Error during SAM2 prediction: {e}", exc_info=True)
            return None

class SAM2OneShotEvaluator:
    """Evaluator for SAM2 One-Shot predictions"""

    def __init__(self, config: Config):
        self.config = config
        self.logger = logging.getLogger("PolypSegmentation.SAM2OneShotEvaluator")
        self.metrics_calc = MetricsCalculator()
        self.wrapper = SAM2OneShotWrapper(config)

    def evaluate(self, test_loader: DataLoader) -> Dict[str, float]:
        """[TRANSLATED] SAM2 [TRANSLATED] [TRANSLATED] [TRANSLATED]"""
        self.logger.info("Evaluating SAM2 One-Shot on test set")
        dice_scores, iou_scores, pixel_accs = [], [], []

        for images, masks in tqdm(test_loader, desc="Testing SAM2 One-Shot"):
            image_np = images[0].cpu().permute(1, 2, 0).numpy()
            mask_np = masks[0, 0].cpu().numpy()

            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            image_rgb = np.clip(std * image_np + mean, 0, 1)
            image_uint8 = (image_rgb * 255).astype(np.uint8)

            # Prediction
            pred_mask = self.wrapper.predict(image_uint8)

            if pred_mask is not None:
                pred_probs = pred_mask.clamp(0, 1)
                pred_logits = torch.logit(pred_probs, eps=1e-6)
            else:
                pred_logits = torch.logit(torch.full_like(masks[0], 1e-6), eps=1e-6)

            gt_mask = masks[0]

            dice_scores.append(self.metrics_calc.dice_coefficient(pred_logits, gt_mask))
            iou_scores.append(self.metrics_calc.iou_score(pred_logits, gt_mask))
            pixel_accs.append(self.metrics_calc.pixel_accuracy(pred_logits, gt_mask))

        return {
            'mean_dice': float(np.mean(dice_scores)),
            'std_dice': float(np.std(dice_scores)),
            'mean_iou': float(np.mean(iou_scores)),
            'std_iou': float(np.std(iou_scores)),
            'mean_pixel_acc': float(np.mean(pixel_accs)),
            'std_pixel_acc': float(np.std(pixel_accs)),
        }

    def visualize_predictions(self, test_loader: DataLoader, num_samples: int = 10):
        """Visualization of SAM2 predictions"""
        self.logger.info("Visualizing SAM2 One-Shot predictions")

        fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
        if num_samples == 1:
            axes = axes.reshape(1, -1)

        col_titles = ["Input Image", "Ground Truth", "SAM2 Prob.", "SAM2 Mask"]
        for j, title in enumerate(col_titles):
            axes[0, j].set_title(title, fontsize=14, fontweight="bold")

        sample_count = 0
        for images, masks in test_loader:
            if sample_count >= num_samples:
                break

            image_np = images[0].cpu().permute(1, 2, 0).numpy()
            mask_np = masks[0, 0].cpu().numpy()

            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            image_rgb = np.clip(std * image_np + mean, 0, 1)
            image_uint8 = (image_rgb * 255).astype(np.uint8)

            pred_mask = self.wrapper.predict(image_uint8)
            if pred_mask is not None:
                pred = pred_mask.squeeze(0).numpy()
            else:
                pred = np.zeros_like(mask_np)

            axes[sample_count, 0].imshow(image_rgb)
            axes[sample_count, 1].imshow(mask_np, cmap='gray')
            axes[sample_count, 2].imshow(pred, cmap='hot', vmin=0, vmax=1)
            axes[sample_count, 3].imshow((pred > 0.5).astype(np.float32), cmap='gray')

            for j in range(4):
                axes[sample_count, j].axis('off')

            sample_count += 1

        plt.tight_layout()
        save_path = self.config.run_dir / "visualizations" / "SAM2_OneShot_predictions.png"
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        self.logger.info(f"Saved SAM2 One-Shot visualization to {save_path}")


This cell defines the CombinedLoss class, which implements a loss function that combines the Dice Loss and Binary Cross-Entropy (BCE) Loss. This combined loss function is commonly used for image segmentation tasks to address challenges like class imbalance and focus on the overlap between predicted and ground truth masks.

Using a combined loss function often leads to better segmentation performance compared to using Dice or BCE alone. Dice Loss helps the model focus on maximizing the overlap, while BCE provides stable gradients, especially for pixels far from the decision boundary. The weighted combination allows tuning the balance between these two objectives.

In [36]:
class CombinedLoss(nn.Module):
    def __init__(self, dice_weight=0.5, bce_weight=0.5):
        super().__init__()
        self.dice_weight, self.bce_weight = dice_weight, bce_weight
        self.bce = nn.BCEWithLogitsLoss()

    def dice_loss(self, logits, target, smooth=1e-7):
        probs = torch.sigmoid(logits)
        intersection = (probs * target).sum(dim=(2, 3))
        union = probs.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        return 1 - ((2. * intersection + smooth) / (union + smooth)).mean()

    def forward(self, logits, target):
        return self.dice_weight * self.dice_loss(logits, target) + self.bce_weight * self.bce(logits, target)

This cell defines the ModelFactory class, which serves as a convenient way to create instances of different segmentation models used in the pipeline based on their names.

In [37]:
class ModelFactory:
    @staticmethod
    def create_unet(config): return UNet(3, 1, [64, 128, 256, 512])
    @staticmethod
    def create_unet_plus(config): return smp.UnetPlusPlus(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1, activation=None)
    @staticmethod
    def create_deeplabv3plus(config): return smp.DeepLabV3Plus(encoder_name="resnet50", encoder_weights="imagenet", in_channels=3, classes=1, activation=None)
    @staticmethod
    def create_fpn(config): return smp.FPN(encoder_name="efficientnet-b3", encoder_weights="imagenet", in_channels=3, classes=1, activation=None)
    @staticmethod
    def create_sam_finetuned(config): return SAMSegmentationWrapper("sam_vit_b_01ec64.pth", "vit_b", True, config, 1024)
    @staticmethod
    def create_yolov8(config): return YOLOSegmentationWrapper("yolov8n-seg", config)
    @staticmethod
    def create_yolov11(config): return YOLOSegmentationWrapper("yolo11n-seg", config)

Purpose: This cell defines the Evaluator class, which is responsible for evaluating the performance of the trained segmentation models on the test dataset and visualizing their predictions.

Evaluating models on a separate test set provides an unbiased estimate of their performance on unseen data. The Evaluator class automates this process and provides both quantitative metrics and qualitative visualizations, which are essential for understanding the strengths and weaknesses of each model and comparing their effectiveness for the polyp segmentation task.


In [38]:
class Evaluator:
    def __init__(self, config):
        self.config = config
        self.logger = logging.getLogger("PolypSegmentation.Evaluator")
        self.metrics_calc = MetricsCalculator()

    def evaluate(self, model, model_name, test_loader):
        model.eval()
        dice_scores, iou_scores, pixel_accs = [], [], []
        with torch.no_grad():
            for images, masks in tqdm(test_loader, desc=f"Testing {model_name}"):
                images, masks = images.to(self.config.device), masks.to(self.config.device)
                outputs = model(images)
                dice_scores.append(self.metrics_calc.dice_coefficient(outputs[0], masks[0]))
                iou_scores.append(self.metrics_calc.iou_score(outputs[0], masks[0]))
                pixel_accs.append(self.metrics_calc.pixel_accuracy(outputs[0], masks[0]))
        return {'mean_dice': np.mean(dice_scores), 'std_dice': np.std(dice_scores),
                'mean_iou': np.mean(iou_scores), 'std_iou': np.std(iou_scores),
                'mean_pixel_acc': np.mean(pixel_accs), 'std_pixel_acc': np.std(pixel_accs)}

    def visualize_predictions(self, model, model_name, test_loader, num_samples=10):
        import torch
        import matplotlib.pyplot as plt
        import numpy as np

        model.eval()
        fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))

        # If num_samples == 1, reshape axes to 2D
        if num_samples == 1:
            axes = axes.reshape(1, -1)

        col_titles = ["Input Image", "Ground Truth", "Predicted (Prob.)", "Predicted Mask"]

        # Add column labels (once at the top)
        for j, title in enumerate(col_titles):
            axes[0, j].set_title(title, fontsize=14, fontweight="bold")

        with torch.no_grad():
            for idx, (images, masks) in enumerate(test_loader):
                if idx >= num_samples:
                    break

                images, masks = images.to(self.config.device), masks.to(self.config.device)
                outputs = model(images)

                image = images[0].cpu().permute(1, 2, 0).numpy()
                mask = masks[0, 0].cpu().numpy()
                pred = torch.sigmoid(outputs[0, 0]).cpu().numpy()

                # Denormalization
                mean = np.array([0.485, 0.456, 0.406])
                std = np.array([0.229, 0.224, 0.225])
                image = np.clip(std * image + mean, 0, 1)

                axes[idx, 0].imshow(image)
                axes[idx, 1].imshow(mask, cmap='gray')
                axes[idx, 2].imshow(pred, cmap='hot', vmin=0, vmax=1)
                axes[idx, 3].imshow((pred > 0.5).astype(np.float32), cmap='gray')

                for j in range(4):
                    axes[idx, j].axis('off')

        plt.tight_layout()
        save_path = self.config.run_dir / "visualizations" / f"{model_name}_predictions.png"
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        self.logger.info(f"Saved prediction visualization to {save_path}")

The ExperimentManager class takes a Config object as input and sets up logging.
- **run** method is the main entry point, executing the pipeline in sequence: data preparation, YOLO dataset conversion, training of various models (UNet, Unet++, DeepLabV3+, FPN, YOLOv8, YOLOv11, fine-tuned SAM), evaluation of trained models, evaluation of SAM One-Shot, evaluation of SAM2 One-Shot, and finally generating a comprehensive report.

- **train_yolo_model** handles the specific training process for YOLO models, which use their native API. It initializes the YOLO wrapper, trainer, calls the fit method, loads the best checkpoint, and then evaluates the model using the YOLOEvaluator.

- **train_model** handles the training of standard PyTorch models (UNet, Unet++, DeepLabV3+, FPN, fine-tuned SAM). It initializes the model using the ModelFactory, sets up the optimizer, scheduler, and loss function, and calls the fit method of the appropriate Trainer class (including the specialized SAMTrainer for fine-tuned SAM). It also adjusts dataloader batch sizes and number of workers for SAM and YOLO models during training.

- **evaluate_sam_one_shot** and **evaluate_sam2_one_shot** specifically handle the evaluation of the one-shot SAM and SAM2 models using their respective evaluators.

- **evaluate_model** loads the best checkpoint for a trained PyTorch model and evaluates it using the standard Evaluator.

- **generate_report** consolidates the training and test results for all models into a pandas DataFrame, saves it to a CSV file, prints a summary, identifies the best performing and fastest models, and generates comparison plots.
create_comparison_plots visualizes the test Dice and IoU scores, training time, and a scatter plot of performance vs. training time using matplotlib.
Implementation details:


In [39]:
class ExperimentManager:
    def __init__(self, config):
        self.config = config
        self.logger = setup_logger(config)
        self.logger.info("="*60 + "\nPOLYP SEGMENTATION PIPELINE\n" + "="*60)
        self.results = {}

    def evaluate_sam_one_shot(self, test_loader):
          evaluator = SAMOneShotEvaluator(self.config)
          test_metrics = evaluator.evaluate(test_loader)
          evaluator.visualize_predictions(test_loader, num_samples=10)

          self.results['SAM_OneShot'] = {
              'training': {'best_dice': test_metrics['mean_dice'], 'training_time': 0.0, 'epochs_trained': 0},
              'test': test_metrics
          }

    def run(self):
        try:
            data_module = DataModule(self.config)
            dataloaders = data_module.prepare_data()

            # Convert dataset to YOLO format for YOLO models
            yolo_converter = YOLODatasetConverter(self.config)
            yolo_data_yaml = yolo_converter.convert_to_yolo_format(
                data_module.train_imgs, data_module.train_masks,
                data_module.val_imgs, data_module.val_masks,
                data_module.test_imgs, data_module.test_masks
            )

            models_to_train = [
                ('UNet_Custom', ModelFactory.create_unet, Trainer, False),
                ('UNetPlusPlus_ResNet34', ModelFactory.create_unet_plus, Trainer, False),
                ('DeepLabV3Plus_ResNet50', ModelFactory.create_deeplabv3plus, Trainer, False),
                ('FPN_EfficientNetB3', ModelFactory.create_fpn, Trainer, False),
                ('YOLOv8n-seg', ModelFactory.create_yolov8, YOLOTrainer, True),
                ('YOLOv11n-seg', ModelFactory.create_yolov11, YOLOTrainer, True),
                ('SAM_ViT-B', ModelFactory.create_sam_finetuned, SAMTrainer, False),
            ]

            for model_name, model_factory, trainer_class, is_yolo in models_to_train:
                if is_yolo:
                    self.train_yolo_model(model_name, model_factory, yolo_data_yaml, dataloaders['test'])
                else:
                    self.train_model(model_name, model_factory, dataloaders, trainer_class)

            for model_name, _, _, is_yolo in models_to_train:
                if not is_yolo:
                    self.evaluate_model(model_name, dataloaders['test'])

            self.evaluate_sam_one_shot(dataloaders['test'])
            self.evaluate_sam2_one_shot(dataloaders['test'])
            self.generate_report()
        except Exception as e:
            self.logger.error(f"Pipeline failed: {e}", exc_info=True)
            raise

    def train_yolo_model(self, model_name, model_factory, yolo_data_yaml, test_loader):
        """Train YOLO model using native API"""
        torch.cuda.empty_cache()

        model_wrapper = model_factory(self.config)
        trainer = YOLOTrainer(model_wrapper, model_name, self.config, yolo_data_yaml)

        training_results = trainer.fit()

        # Evaluate
        trainer.load_checkpoint('best')
        evaluator = YOLOEvaluator(self.config)
        test_metrics = evaluator.evaluate(model_wrapper, model_name, test_loader)
        evaluator.visualize_predictions(model_wrapper, model_name, test_loader, num_samples=10)

        self.results[model_name] = {
            'training': training_results,
            'trainer': trainer,
            'test': test_metrics
        }

        torch.cuda.empty_cache()

    def train_model(self, model_name, model_factory, dataloaders, trainer_class):
        torch.cuda.empty_cache()
        model = model_factory(self.config)

        if 'SAM' in model_name:
            train_loader = DataLoader(dataloaders['train'].dataset, batch_size=1, shuffle=True, num_workers=0, pin_memory=False)
            val_loader = DataLoader(dataloaders['val'].dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=False)
            dataloaders_adj = {'train': train_loader, 'val': val_loader, 'test': dataloaders['test']}
            grad_accum = 8
        elif 'YOLO' in model_name or 'yolo' in model_name:
            train_loader = DataLoader(dataloaders['train'].dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
            val_loader = DataLoader(dataloaders['val'].dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
            dataloaders_adj = {'train': train_loader, 'val': val_loader, 'test': dataloaders['test']}
            grad_accum = 1
        else:
            dataloaders_adj = dataloaders
            grad_accum = 1

        optimizer = AdamW(model.parameters(), lr=self.config.learning_rate, weight_decay=self.config.weight_decay)
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
        criterion = CombinedLoss(dice_weight=0.7, bce_weight=0.3)

        trainer = (trainer_class(model, model_name, self.config, criterion, optimizer, scheduler, grad_accum, True)
                  if trainer_class == SAMTrainer else
                  trainer_class(model, model_name, self.config, criterion, optimizer, scheduler))

        training_results = trainer.fit(dataloaders_adj['train'], dataloaders_adj['val'])
        self.results[model_name] = {'training': training_results, 'trainer': trainer}
        torch.cuda.empty_cache()


    def evaluate_sam2_one_shot(self, test_loader):
        """Evaluation of SAM2 in one-shot mode"""
        evaluator = SAM2OneShotEvaluator(self.config)
        test_metrics = evaluator.evaluate(test_loader)
        evaluator.visualize_predictions(test_loader, num_samples=10)

        self.results['SAM2_OneShot'] = {
            'training': {
                'best_dice': test_metrics['mean_dice'],
                'training_time': 0.0,
                'epochs_trained': 0
            },
            'test': test_metrics
        }

    def evaluate_model(self, model_name, test_loader):
        trainer = self.results[model_name]['trainer']
        trainer.load_checkpoint('best')
        evaluator = Evaluator(self.config)
        test_metrics = evaluator.evaluate(trainer.model, model_name, test_loader)
        evaluator.visualize_predictions(trainer.model, model_name, test_loader, num_samples=10)
        self.results[model_name]['test'] = test_metrics

    def generate_report(self):
        self.logger.info("="*60 + "\nGENERATING FINAL REPORT\n" + "="*60)
        report_data = []
        for model_name, results in self.results.items():
            report_data.append({
                'Model': model_name,
                'Best Val Dice': results['training']['best_dice'],
                'Test Dice': results['test']['mean_dice'],
                'Test Dice Std': results['test']['std_dice'],
                'Test IoU': results['test']['mean_iou'],
                'Test IoU Std': results['test']['std_iou'],
                'Test Pixel Acc': results['test']['mean_pixel_acc'],
                'Training Time (s)': results['training']['training_time'],
                'Epochs Trained': results['training']['epochs_trained'],
                'Passed Threshold': '✓' if results['test']['mean_dice'] >= self.config.dice_threshold else '✗'
            })

        df_report = pd.DataFrame(report_data).sort_values('Test Dice', ascending=False)
        report_path = self.config.run_dir / "metrics" / "final_report.csv"
        df_report.to_csv(report_path, index=False)

        self.logger.info("\n" + "="*80 + "\nFINAL RESULTS SUMMARY\n" + "="*80)
        self.logger.info("\n" + df_report.to_string(index=False) + "\n" + "="*80)

        best_model = df_report.iloc[0]['Model']
        best_dice = df_report.iloc[0]['Test Dice']
        self.logger.info(f"\nBest Model: {best_model}\n   Test Dice Score: {best_dice:.4f}")

        fastest_model = df_report.loc[df_report['Training Time (s)'].idxmin(), 'Model']
        fastest_time = df_report['Training Time (s)'].min()
        self.logger.info(f"\n⚡ Fastest Training: {fastest_model}\n   Training Time: {fastest_time:.2f}s")

        self.create_comparison_plots(df_report)
        self.logger.info(f"\nFull report saved to: {report_path}")

    def create_comparison_plots(self, df_report):
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))

        axes[0, 0].bar(df_report['Model'], df_report['Test Dice'], color='steelblue')
        axes[0, 0].axhline(y=self.config.dice_threshold, color='r', linestyle='--', label='Threshold')
        axes[0, 0].set_ylabel('Dice Score')
        axes[0, 0].set_title('Test Dice Score by Model')
        axes[0, 0].tick_params(axis='x', rotation=45)
        axes[0, 0].legend()
        axes[0, 0].grid(axis='y', alpha=0.3)

        axes[0, 1].bar(df_report['Model'], df_report['Test IoU'], color='seagreen')
        axes[0, 1].set_ylabel('IoU Score')
        axes[0, 1].set_title('Test IoU Score by Model')
        axes[0, 1].tick_params(axis='x', rotation=45)
        axes[0, 1].grid(axis='y', alpha=0.3)

        axes[1, 0].bar(df_report['Model'], df_report['Training Time (s)'], color='coral')
        axes[1, 0].set_ylabel('Time (seconds)')
        axes[1, 0].set_title('Training Time by Model')
        axes[1, 0].tick_params(axis='x', rotation=45)
        axes[1, 0].grid(axis='y', alpha=0.3)

        axes[1, 1].scatter(df_report['Training Time (s)'], df_report['Test Dice'],
                          s=200, alpha=0.6, c=range(len(df_report)), cmap='viridis')
        for idx, row in df_report.iterrows():
            axes[1, 1].annotate(row['Model'].replace('_', '\n'),
                               (row['Training Time (s)'], row['Test Dice']), fontsize=8, ha='center')
        axes[1, 1].set_xlabel('Training Time (s)')
        axes[1, 1].set_ylabel('Test Dice Score')
        axes[1, 1].set_title('Performance vs Training Time')
        axes[1, 1].grid(alpha=0.3)

        plt.tight_layout()
        save_path = self.config.run_dir / "visualizations" / "model_comparison.png"
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        self.logger.info(f"Saved comparison plots to {save_path}")


In [ ]:
def main():
    """Main execution function"""
    config = Config(
        dataset_path="/content/drive/MyDrive/Kvasir-Capsule",
        output_dir="./output",
        batch_size=32,
        num_epochs=50,
        learning_rate=1e-4,
        early_stopping_patience=15,
        img_size=(256, 256),
        num_workers=0
    )

    experiment = ExperimentManager(config)
    experiment.run()

    print("\n" + "="*80)
    print("Pipeline completed successfully!")
    print(f"Results saved to: {config.run_dir}")
    print("="*80)

if __name__ == "__main__":
    main()

In [41]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Conclusions

### Models of the YOLO family:
The predictions of these models look rather noisy, although it is clear that the model captures the features that it needs to find. Provided that the dataset is very small, this is a good result. Perhaps if we add proper post-processing, we can achieve more or less good results. But in the medical field, where accuracy is important, such models are not very suitable yet.
It is also clear that the new generation YOLOv11 performed noticeably worse, but how can it be that newer versions of models work worse than time-tested models? Here's what it might be related to:
1. YOLO11n is a very lightweight model. The new C3k2 and C2PSA blocks were optimized primarily for detection/tracking, rather than segmentation.
2. The official Ultralytics (Instance Segmentation) documentation emphasizes that additional architecture and hyperparameters may be required to specialize in specific tasks. On a small dataset (35 training images), more "rigid" regularization and lower YOLO11n capacity lead to under-training of masks.
3. The YOLO11n segmentation head is "thinner" and more dependent on fine-tuning
4. Due to the differences in architecture and size of the model, YOLO11n often needs longer fine-tuning:
- more epochs,
- increased imgsz
- loosening close_mosaic
    - or an obvious decrease (lr0)

    All this is reflected in the documentation (Hyperparameters). On small datasets, patience=0 (lengthen learning) and disabling aggressive augmentation also helps.

### SAM_ViT-B
Next, a few words about the work of the SAM_ViT-B model. In the Predictions, it can be seen that the predicted masks have a ragged (noisy) character, and often the model fills in unnecessary areas with the mask. And here are some thoughts on why there might be:
1. In our code, SAM is used as a pure encoder, and the mask is built by a separate head. In other words, we actually use only high-level features of the pre-trained model, for which SAM usually uses its own decoders and iterative postprocessing.
2. Adapter - just two 3x3 convolutions+BatchNorm+Dropout. For clear boundaries, multi-scale features, skip connections, or a more complex head (FPN, UPerNet, ASPP) are needed.
3. The data reader gives 256x256, in SAMwrapper we scale to 1024, SAM internally works on 64x64 feature map, then scale down again.
The result: a "stepped" contour, smoothing, noise. The lower the input resolution, the more artifacts there are after two interpolations.

### DeepLabV3Plus_ResNet50 and UNetPlusPlus_ResNet34
These models performed the best of all. I think with the advent of new data in the dataset, the results will be suitable for use in real situations.

### One-Shot solutions
These solutions did not show a proper solution, although the sam and sam2 models in the initial dataset had a large pool of medical data. We see that the resulting masks simply fill in almost the entire visible range of the image.
Below are the possible causes:
1. Domain feature - the background (mucosa) has a similar texture to the polyp.
2. For SAM, who has not undergone fine-tuning or full-fledged adaptation, a small pro-mask without negative signals leads to "merging" with the background.
3. It is necessary to do a preliminary limitation of the area of attention (for example, take a bounding box from a simple detector).